---
# **Lab: Graph Neural Networks (GNN)**
---

# ▶️ CUDA tools...

In [ ]:
!nvidia-smi

# ✅ GCN on Cora Dataset

📘 **Cora Dataset — Overview**

The Cora dataset is one of the most widely used citation networks for evaluating graph neural networks (GNNs), especially for semi-supervised node classification.

It consists of:
-	**Nodes**: scientific publications
-	**Edges**: citation links between publications
-	**Node** **features**: bag-of-words vectors
-	**Node labels**: research topic of the publication


📊 **Dataset Statistics**

Typical version used in GNN papers and PyTorch Geometric (Planetoid/Cora):

|Property|	Value|
|----|----|
|Number of nodes|	2,708|
|Number of edges|	5,429 (undirected)|
|Number of classes|	7|
|Feature dimension|	1,433|
|Train nodes|	140|
|Validation nodes|	500|
|Test nodes|	1,000|


🧠 **Graph Structure**

**Nodes**

Each node represents a machine learning publication.

**Edges**

An edge exists if paper A cites paper B.
Although citation is directed, the version used in GNN papers treats the graph as undirected for simplicity.

**Node Features**

Each publication is represented by a 1,433-dimensional binary bag-of-words feature vector:
-	Feature = 1 → the corresponding word appears in the paper
-	Feature = 0 → the word does not appear

These features form the input to the GCN.

**Labels**

Each publication belongs to one of 7 topics:
1.	Case-based
2.	Genetic Algorithms
3.	Neural Networks
4.	Probabilistic Methods
5.	Reinforcement Learning
6.	Rule Learning
7.	Theory

🎯 **Task: Semi-Supervised Node Classification**

Cora is used for transductive learning:
-	You are given the entire graph structure
-	Only a small subset of nodes (140) have labels
-	The goal is to infer the labels of the remaining 2,568 unlabeled nodes

This setup matches real-world problems like classifying users in a social network where only some accounts are labeled.


🏗️ **Why Cora Is Important**

Cora was the original dataset used in the seminal paper:

Kipf & Welling (2017). “Semi-Supervised Classification with Graph Convolutional Networks.”

It is still a standard benchmark for comparing: GCN, GraphSAGE, GAT, APPNP, GIN, Message-passing networks, Graph transformers


🔧 **How PyTorch Geometric Loads Cora**

Via `PyG` library installation:

```bash
pip install torch torch_geometric
```

`PyG` provides it via:

```
from torch_geometric.datasets import Planetoid
dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]
````

data contains:
-	x:          node features [2708, 1433]
-	edge_index: edges [2, 5429]
-	y:          labels [2708]
-	train_mask, val_mask, test_mask

Cora graph visualization:

In [ ]:
import math
import matplotlib.pyplot as plt
import networkx as nx
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_networkx

# ---------------------------
# Load Cora dataset
# ---------------------------
dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0]

# Convert to NetworkX graph
G = to_networkx(data, to_undirected=True)

# Node colors = class labels
labels = data.y.numpy()
node_colors = [labels[i] for i in range(data.num_nodes)]

# ---------------------------
# Compute graph layout
# ---------------------------
print("Computing graph layout... (this may take a few seconds)")
pos = nx.spring_layout(G, seed=42, k=3 / math.sqrt(data.num_nodes))

# ---------------------------
# Plot full graph
# ---------------------------
plt.figure(figsize=(14, 10))
nx.draw_networkx_nodes(
    G, pos, node_size=20, node_color=node_colors, cmap=plt.cm.tab10, alpha=0.85
)
nx.draw_networkx_edges(G, pos, width=0.3, alpha=0.2)

plt.title("Full Cora Citation Network (Nodes colored by class)")
plt.axis("off")
plt.show()

**Use GCNLayer to work on Cora for node classification** 

Below is a complete, minimal training demo using:
-	Your GCNLayer as-is
-	PyTorch Geometric only for loading the Cora dataset
-	A simple 2-layer GCN + training loop

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import Planetoid

# ---------------------------
# utils
# ---------------------------

def load_cora():
    dataset = Planetoid(root="data/Cora", name="Cora")
    data = dataset[0]  # Single graph
    
    print("Num nodes:", data.num_nodes)
    print("Num features:", dataset.num_features)
    print("Num classes:", dataset.num_classes)

    return dataset, data

def build_adj_matrix(data):
    
    num_nodes = data.num_nodes
    edge_index = data.edge_index  # [2, E]

    # Create adjacency matrix A: [N, N]
    adj = torch.zeros((num_nodes, num_nodes), dtype=torch.float32)

    # Add edges
    adj[edge_index[0], edge_index[1]] = 1.0

    # Make sure it's symmetric (Cora is undirected, but we enforce it)
    adj = torch.maximum(adj, adj.t())

    # Add self-loops (identity)
    adj = adj + torch.eye(num_nodes)

    # Add batch dimension for your layer: [1, N, N]
    adj = adj.unsqueeze(0)
    return adj

def accuracy(logits, labels, mask):
    preds = logits.argmax(dim=-1)
    correct = (preds[mask] == labels[mask]).sum().item()
    total = int(mask.sum())
    return correct / total if total > 0 else 0.0

# ---------------------------
# The GCN model
# ---------------------------
class GCNLayer(nn.Module):
    def __init__(self, c_in, c_out):
        super().__init__()
        self.projection = nn.Linear(c_in, c_out)

    def forward(self, node_feats, adj_matrix):
        """
        node_feats: [batch_size, num_nodes, c_in]
        adj_matrix: [batch_size, num_nodes, num_nodes]
                    Assumes self-loops already added.
        """
        # Num neighbours = number of incoming edges
        num_neighbours = adj_matrix.sum(dim=-1, keepdims=True)  # [batch, num_nodes, 1]

        # Linear projection
        node_feats = self.projection(node_feats)  # [batch, num_nodes, c_out]

        # Aggregate neighbour features
        node_feats = torch.bmm(adj_matrix, node_feats)  # [batch, num_nodes, c_out]

        # Mean over neighbours
        node_feats = node_feats / num_neighbours.clamp(min=1.0)

        return node_feats

class GCNNet(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim):
        super().__init__()
        self.gcn1 = GCNLayer(in_dim, hidden_dim)
        self.gcn2 = GCNLayer(hidden_dim, out_dim)

    def forward(self, x, adj):
        # x: [1, N, F], adj: [1, N, N]
        x = self.gcn1(x, adj)  # [1, N, H]
        x = F.relu(x)
        x = self.gcn2(x, adj)  # [1, N, C]
        return x.squeeze(0)  # [N, C] (drop batch dim for loss/metrics)

# ---------------------------   
# Setup and training
# ---------------------------

# Load Cora with PyG
dataset, data = load_cora()

# Build adjacency matrix
adj = build_adj_matrix(data)

# Node features: [N, F]
x = data.x  # float tensor
y = data.y  # long tensor of labels

# Also add batch dimension to node features: [1, N, F]
x = x.unsqueeze(0)

# Train/val/test masks: [N]
train_mask = data.train_mask  # boolean tensor of 104 positive entries
val_mask = data.val_mask      # boolean tensor of 500 positive entries
test_mask = data.test_mask    # boolean tensor of 1000 positive entries

# the model 
model = GCNNet(
    in_dim=dataset.num_features,
    hidden_dim=16,
    out_dim=dataset.num_classes
)

# ---------------------------
# Move to GPU if available
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
x = x.to(device)
adj = adj.to(device)
y = y.to(device)
train_mask = train_mask.to(device)
val_mask = val_mask.to(device)
test_mask = test_mask.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = nn.CrossEntropyLoss()

# ---------------------------
# Training loop
# ---------------------------

for epoch in range(1, 201):
    model.train()
    optimizer.zero_grad()

    out = model(x, adj)  # [N, num_classes]
    loss = criterion(out[train_mask], y[train_mask])
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        out_eval = model(x, adj)
        train_acc = accuracy(out_eval, y, train_mask)
        val_acc = accuracy(out_eval, y, val_mask)
        test_acc = accuracy(out_eval, y, test_mask)

    if epoch % 20 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:03d} | "
            f"Loss: {loss.item():.4f} | "
            f"Train Acc: {train_acc:.4f} | "
            f"Val Acc: {val_acc:.4f} | "
            f"Test Acc: {test_acc:.4f}"
        )

print("Final Test Accuracy:", test_acc)


# ✅ Using PyG Framework

#### **PyTorch Geometric (PyG)** is a library built on top of PyTorch for deep learning on graphs.

It provides:

- Efficient sparse graph operations  
- Message-passing abstractions  
- Ready-to-use benchmark datasets (Cora, CiteSeer, PubMed)  
- Many GNN layers and utilities  

Core idea:

All GNN layers follow a **message passing framework**:

$$
\text{Message} \rightarrow \text{Aggregate} \rightarrow \text{Update}
$$

---

#### 1️⃣ GCN (Graph Convolutional Network)

**Operator:** `GCNConv`

Update rule:

$$
X' = \hat{D}^{-1/2}\hat{A}\hat{D}^{-1/2}X\Theta
$$

Properties:

- Symmetric degree normalization  
- Strong smoothing effect  
- Classic baseline for citation networks  

Good for:
- Transductive learning (e.g., Cora)

---

#### 2️⃣ GraphConv

**Operator:** `GraphConv`

Update rule:

$$
x_i' = W_1 x_i + W_2 \sum_{j \in \mathcal{N}(i)} x_j
$$

Properties:

- Separate transforms for self and neighbors  
- Flexible aggregation (`add`, `mean`, `max`)  
- No built-in symmetric normalization  

More flexible than GCN.

---

#### 3️⃣ GraphSAGE

**Operator:** `SAGEConv`

Update rule:

$$
x_i' = W_1 x_i + W_2 \,\text{mean}_{j \in \mathcal{N}(i)} x_j
$$

Properties:

- Designed for **inductive learning**
- Aggregates neighbor information explicitly  
- Can generalize to unseen graphs  

Common in large-scale graph learning.

---

#### Summary

| Model       | Normalization | Self/Neighbor Separation | Inductive Friendly |
|-------------|--------------|--------------------------|--------------------|
| GCN         | Yes          | No                       | Moderate           |
| GraphConv   | No           | Yes                      | Moderate           |
| GraphSAGE   | No           | Yes                      | Yes                |

All three follow the same message-passing principle,
but differ in **aggregation and inductive bias**.

---

In [ ]:
import torch
import torch.nn.functional as F
from torch_geometric.datasets import Planetoid
from torch_geometric.nn import GCNConv, GraphConv, SAGEConv

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

dataset = Planetoid(root="data/Cora", name="Cora")
data = dataset[0].to(device)

print("Cora:", data)
print("Nodes:", data.num_nodes,
      "Edges:", data.num_edges,
      "Features:", dataset.num_features,
      "Classes:", dataset.num_classes)

**GCN_Model implements a 2-layer Graph Convolutional Network (GCN) using PyTorch Geometric**.

It performs node classification on a graph (e.g., Cora or CiteSeer).

- conv1
	-	Takes input node features of size in_dim
	-	Produces hidden embeddings of size hidden_dim
- conv2
	-	Takes hidden embeddings
	-	Produces output of size out_dim
	-	out_dim = number of classes

So the architecture is:
$$
\text{Input} \rightarrow \text{GCNConv} \rightarrow \text{ReLU} \rightarrow \text{Dropout} \rightarrow \text{GCNConv}
$$

-  forward:
	-	`x`: node feature matrix [N, F]
	-	`edge_index`: sparse graph connectivity [2, E]
- Dropout:
    - Regularization technique
	- Prevents overfitting
	- Active only during training


In [ ]:
class GCN_Model(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout_p=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim) # alternative to GraphConv
        self.conv2 = GCNConv(hidden_dim, out_dim) # alternative to GraphConv
        self.dropout_p = dropout_p

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout_p, training=self.training)
        x = self.conv2(x, edge_index)
        return x  # logits [N, C]

In [ ]:
# Accuracy function
@torch.no_grad()
def accuracy(logits, y, mask):
    pred = logits.argmax(dim=1)
    correct = (pred[mask] == y[mask]).sum().item()
    return correct / int(mask.sum())

# ---------------------------
# Training and evaluation functions
# ---------------------------
def train_one_epoch(model, optimizer, data):
    model.train()
    optimizer.zero_grad()
    logits = model(data)
    loss = F.cross_entropy(logits[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()

# Evaluation function
@torch.no_grad()
def evaluate(model, data):
    model.eval()
    logits = model(data)
    train_acc = accuracy(logits, data.y, data.train_mask)
    val_acc   = accuracy(logits, data.y, data.val_mask)
    test_acc  = accuracy(logits, data.y, data.test_mask)
    return train_acc, val_acc, test_acc

In [ ]:
def run_experiment(model, data, epochs=200, lr=0.01, weight_decay=5e-4):
    """
    Train and evaluate the model, returning best validation and corresponding test accuracy

        Parameters:
        - model: the GNN model to train
        - data: the PyG data object containing the graph and masks
        - epochs: number of training epochs
        - lr: learning rate for the optimizer
        - weight_decay: L2 regularization strength for the optimizer
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    best_val = 0.0
    best_test = 0.0

    for epoch in range(1, epochs + 1):
        loss = train_one_epoch(model, optimizer, data)
        train_acc, val_acc, test_acc = evaluate(model, data)

        if val_acc > best_val:
            best_val = val_acc
            best_test = test_acc

        if epoch % 20 == 0 or epoch == 1:
            print(f"Epoch {epoch:03d} | Loss {loss:.4f} | "
                  f"Train {train_acc:.4f} | Val {val_acc:.4f} | Test {test_acc:.4f}")

    print(f"Best Val Acc: {best_val:.4f} | Test @ Best Val: {best_test:.4f}")
    return best_val, best_test

In [ ]:
print("\n=== Model A: GCNConv ===")
gcn = GCN_Model(dataset.num_features, 16, dataset.num_classes, dropout_p=0.5)
best_val, best_test = run_experiment(gcn, data)

## ↘️ TODO...

**Objective: Apply and Compare GCNConv vs GraphConv**

Train and compare two models on the Cora dataset:

1. **GCNConv model**
2. **GraphConv model**

Understand how architectural differences affect performance

In [ ]:
class GraphConv_Model(torch.nn.Module):
    pass

class GraphSAGE_Model(torch.nn.Module):
    pass

In [ ]:
print("\n=== Model B: GraphConv ===")

print("\n=== Model C: GraphSAGE (SAGEConv) ===")

## ➡️ Solution...

In [ ]:
class GraphConv_Model(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout_p=0.5, aggr="add"):
        super().__init__()
        self.conv1 = GraphConv(in_dim, hidden_dim, aggr=aggr)
        self.conv2 = GraphConv(hidden_dim, out_dim, aggr=aggr)
        self.dropout_p = dropout_p

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout_p, training=self.training)
        x = self.conv2(x, edge_index)
        return x  # logits [N, C]

In [ ]:
class GraphSAGE_Model(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout_p=0.5, aggr="mean"):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden_dim, aggr=aggr)
        self.conv2 = SAGEConv(hidden_dim, out_dim, aggr=aggr)
        self.dropout_p = dropout_p

    def forward(self, data):
        x, edge_index = data.x, data.edge_index
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        #x = F.dropout(x, p=self.dropout_p, training=self.training)
        x = self.conv2(x, edge_index)
        return x  # logits [N, C]

In [ ]:
print("\n=== Model B: GraphConv ===")
graphconv = GraphConv_Model(dataset.num_features, 32, dataset.num_classes, dropout_p=0.5, aggr="add")
run_experiment(graphconv, data)

print("\n=== Model C: GraphSAGE (SAGEConv) ===")
sage = GraphSAGE_Model(dataset.num_features, 32, dataset.num_classes, dropout_p=0.5, aggr="mean")
run_experiment(sage, data);

# ✅ Graph classification

**TUDataset in PyTorch Geometric**


**TUDataset** is a collection of benchmark datasets for **graph classification tasks**. It is commonly used in research on Graph Neural Networks (GNNs).

The datasets originate from the TU Dortmund University repository andare integrated into PyTorch Geometric (PyG) through:

``` python
from torch_geometric.datasets import TUDataset
```

Each dataset consists of **multiple independent graphs**, each with:

-   Node features
-   Edge connections
-   A graph-level label

-  **Structure of TUDataset**

Each dataset in TUDataset contains:

-   A set of graphs: $(G_1, G_2,\dots , G_N )$
-   Each graph $ G_i = (V_i, E_i)$
-   Node feature matrix $X_i \in\mathbb{R}^{\|V_i\|\times F}$
-   Graph label $y_i$

Key characteristics:

-   Graph sizes vary (different numbers of nodes and edges per graph)
-   Labels are assigned **per graph**, not per node
-   Suitable for **inductive learning**

------------------------------------------------------------------------

- **Commonly Used TUDatasets**

  |Dataset       |Task Type              | Classes |  Domain|
  |------------- |----------------------- |--------- |-----------|
  |MUTAG         |Binary classification   |2         |Molecules|
  |ENZYMES       |Multi-class (6)         |6         |Proteins|
  |PROTEINS      |Binary classification   |2         |Proteins|
  |IMDB-BINARY   |Binary classification   |2         |Social networks|
  |COLLAB        |Multi-class             |3         |Social networks|

------------------------------------------------------------------------

- **Example Usage**

``` python
dataset = TUDataset(root="data/TUDataset", name="ENZYMES")
print(len(dataset))                 # Number of graphs
print(dataset.num_features)         # Node feature dimension
print(dataset.num_classes)          # Number of classes
```

Each element of the dataset is a `Data` object containing:

-   `data.x` → Node features
-   `data.edge_index` → Edge list (COO format)
-   `data.y` → Graph label
-   `data.batch` → Batch assignment when using DataLoader

------------------------------------------------------------------------

- **Why TUDataset is Important**

TUDataset is widely used for:

-   Benchmarking graph classification models
-   Comparing different GNN architectures
-   Evaluating pooling and readout strategies
-   Testing inductive generalization

It is especially useful for studying:

-   Graph-level prediction
-   Global pooling strategies
-   Model expressiveness (e.g., GCN vs GIN vs GraphSAGE)

------------------------------------------------------------------------

- **Learning Setting**

TUDataset tasks are **inductive**:

-   Training and test graphs are separate
-   Graph structure of test graphs is unseen during training
-   The model must generalize to entirely new graphs

------------------------------------------------------------------------

- **Summary**

TUDataset provides a standardized collection of graph classification
benchmarks for evaluating GNN models.

It supports research in:

-   Molecular property prediction
-   Bioinformatics
-   Social network analysis
-   Structural graph learning


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool

# ---------------------------
# Dataset: multiple graphs
# --------------------------- 
dataset = TUDataset(root="data/TUDataset", name="MUTAG")  # binary graph classification
dataset = dataset.shuffle()

# Train/val/test split
n = len(dataset)
train_dataset = dataset[: int(0.8 * n)]
val_dataset   = dataset[int(0.8 * n) : int(0.9 * n)]
test_dataset  = dataset[int(0.9 * n) :]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64)
test_loader  = DataLoader(test_dataset, batch_size=64)

print("Graphs:", len(dataset))
print("Node features:", dataset.num_features)
print("Classes:", dataset.num_classes)

# ---------------------------
# Model: GCN + Readout + MLP
# ---------------------------
class GraphGCN(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout_p=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.dropout_p = dropout_p
        self.classifier = nn.Linear(hidden_dim, out_dim)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        # Node embeddings
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout_p, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)

        # Graph readout (invariant to node ordering)
        g = global_mean_pool(x, batch)  # [num_graphs_in_batch, hidden_dim]

        # Graph classification
        out = self.classifier(g)        # [num_graphs_in_batch, num_classes]
        return out


# ---------------------------
# Train / Evaluate
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GraphGCN(dataset.num_features, hidden_dim=64, out_dim=dataset.num_classes).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate(loader):
    model.eval()
    total, correct, total_loss = 0, 0, 0.0
    for data in loader:
        data = data.to(device)
        logits = model(data)
        loss = criterion(logits, data.y)
        total_loss += float(loss.item()) * data.num_graphs

        pred = logits.argmax(dim=1)
        correct += int((pred == data.y).sum())
        total += int(data.num_graphs)
    return total_loss / total, correct / total

def train_one_epoch():
    model.train()
    total_loss, total = 0.0, 0
    for data in train_loader:
        data = data.to(device)
        optimizer.zero_grad()
        logits = model(data)
        loss = criterion(logits, data.y)
        loss.backward()
        optimizer.step()
        total_loss += float(loss.item()) * data.num_graphs
        total += int(data.num_graphs)
    return total_loss / total

best_val_acc = 0.0
best_test_acc = 0.0

for epoch in range(1, 101):
    train_loss = train_one_epoch()
    val_loss, val_acc = evaluate(val_loader)
    test_loss, test_acc = evaluate(test_loader)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_test_acc = test_acc

    if epoch % 10 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:03d} | "
            f"Train loss {train_loss:.4f} | "
            f"Val acc {val_acc:.4f} | "
            f"Test acc {test_acc:.4f}"
        )

print(f"Best Val acc: {best_val_acc:.4f} | Test acc @ best Val: {best_test_acc:.4f}")

## ↘️ TODO...

**Classification on ENZYMES with GCN**

- Train and analyze a Graph Neural Network (GCN) for graph classification on the ENZYMES dataset

1.  Load the ENZYMES dataset:

    ``` python
    dataset = TUDataset(root="data/TUDataset", name="ENZYMES")
    ```

2.  Answer:
    -   How many graphs are in the dataset?
    -   How many node features per graph?
    -   How many classes?
    -   Are all graphs the same size?


🔹 **Train the Given Model**

-   Use the provided 3-layer GCN model with:
    -   Hidden dimension = 64
    -   Dropout = 0.5
    -   Learning rate = 1e-3
    -   Weight decay = 1e-4
    -   200 epochs

🔹 **Tasks**

1.  Track:
    -   Training loss
    -   Validation accuracy
    -   Test accuracy
2.  Record:
    -   Best validation accuracy
    -   Corresponding test accuracy

🔹 **Analyze the Readout Layer**

-   The model uses:
    - `g = global_mean_pool(h^(L))`

🔹 **Modify the Model**

- Pooling
    - Replace: `global_mean_pool`
    - with: `global_add_pool` - `global_max_pool`

-  Depth
    - Modify the model: 
        - 2 layers instead of 3 
        - 4 layers instead of 3

- Replace Convolution
    - Replace GCNConv with: 
        - GraphConv 
        - SAGEConv 
        - GraphSAGE      

## ➡️ Solution...

In [ ]:
## Extend the Graph Classification Model to **ENZYMES** (PyG)

### 1) Full working script (ENZYMES)

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch_geometric.datasets import TUDataset
from torch_geometric.loader import DataLoader
from torch_geometric.nn import GCNConv, global_mean_pool, BatchNorm


# ---------------------------
# Dataset: ENZYMES (many graphs)
# ---------------------------
dataset = TUDataset(root="data/TUDataset", name="ENZYMES")
dataset = dataset.shuffle()

print("Graphs:", len(dataset))
print("Node features:", dataset.num_features)
print("Classes:", dataset.num_classes)

# Train/val/test split (simple random split)
n = len(dataset)
train_dataset = dataset[: int(0.8 * n)]
val_dataset   = dataset[int(0.8 * n) : int(0.9 * n)]
test_dataset  = dataset[int(0.9 * n) :]

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=64)
test_loader  = DataLoader(test_dataset, batch_size=64)


# ---------------------------
# Model: deeper GCN + BN + readout
# ---------------------------
class GraphGCN_ENZYMES(nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim, dropout_p=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.bn1   = BatchNorm(hidden_dim)

        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.bn2   = BatchNorm(hidden_dim)

        self.conv3 = GCNConv(hidden_dim, hidden_dim)
        self.bn3   = BatchNorm(hidden_dim)

        self.dropout_p = dropout_p
        self.classifier = nn.Linear(hidden_dim, out_dim)

    def forward(self, data):
        x, edge_index, batch = data.x, data.edge_index, data.batch

        x = self.conv1(x, edge_index)
        x = self.bn1(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout_p, training=self.training)

        x = self.conv2(x, edge_index)
        x = self.bn2(x)
        x = F.relu(x)
        x = F.dropout(x, p=self.dropout_p, training=self.training)

        x = self.conv3(x, edge_index)
        x = self.bn3(x)
        x = F.relu(x)

        # Graph-level invariant readout
        g = global_mean_pool(x, batch)  # [num_graphs_in_batch, hidden_dim]

        # Graph classification logits
        out = self.classifier(g)        # [num_graphs_in_batch, out_dim]
        return out


# ---------------------------
# Train / Evaluate
# ---------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = GraphGCN_ENZYMES(
    in_dim=dataset.num_features,
    hidden_dim=64,
    out_dim=dataset.num_classes,
    dropout_p=0.5
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = nn.CrossEntropyLoss()


@torch.no_grad()
def evaluate(loader):
    model.eval()
    total, correct, total_loss = 0, 0, 0.0

    for batch_data in loader:
        batch_data = batch_data.to(device)
        logits = model(batch_data)               # [B, C]
        loss = criterion(logits, batch_data.y)   # graph labels

        total_loss += float(loss.item()) * batch_data.num_graphs
        pred = logits.argmax(dim=1)
        correct += int((pred == batch_data.y).sum())
        total += int(batch_data.num_graphs)

    return total_loss / total, correct / total


def train_one_epoch():
    model.train()
    total_loss, total = 0.0, 0

    for batch_data in train_loader:
        batch_data = batch_data.to(device)
        optimizer.zero_grad()
        logits = model(batch_data)
        loss = criterion(logits, batch_data.y)
        loss.backward()
        optimizer.step()

        total_loss += float(loss.item()) * batch_data.num_graphs
        total += int(batch_data.num_graphs)

    return total_loss / total


best_val_acc = 0.0
best_test_acc = 0.0

for epoch in range(1, 201):
    train_loss = train_one_epoch()
    val_loss, val_acc = evaluate(val_loader)
    test_loss, test_acc = evaluate(test_loader)

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_test_acc = test_acc

    if epoch % 20 == 0 or epoch == 1:
        print(
            f"Epoch {epoch:03d} | "
            f"Train loss {train_loss:.4f} | "
            f"Val acc {val_acc:.4f} | "
            f"Test acc {test_acc:.4f}"
        )

print(f"Best Val acc: {best_val_acc:.4f} | Test acc @ best Val: {best_test_acc:.4f}")